# Uber Fare Prediction
**Goal:** Predict the price of an Uber ride from a given pickup point to the drop-off location.

**Tasks**
1. Pre-process the dataset
2. Identify outliers
3. Check the correlation
4. Implement Linear Regression and Random Forest Regression
5. Evaluate both models (R², MAE, MSE, RMSE) and compare them

> Place `uber.csv` in the same folder as this notebook before running.

## 0. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
RANDOM_STATE = 42

## 1. Load the dataset

In [ ]:
uber = pd.read_csv("uber.csv")
print("Shape:", uber.shape)
uber.head()

In [ ]:
uber.info()

In [ ]:
uber.describe().T

## 2. Pre-processing

Steps: remove useless columns → handle missing values → remove duplicates → fix data types → remove impossible values (wrong coordinates, zero/negative fares, invalid passenger counts) → engineer features (trip **distance** using the Haversine formula, plus date/time features).

In [ ]:
# 2.1 Missing values
print("Missing values per column:")
print(uber.isnull().sum())

In [ ]:
# 2.2 Drop index-like columns, drop missing rows and duplicates
cols_to_drop = [c for c in ["Unnamed: 0", "key"] if c in uber.columns]
df = uber.drop(columns=cols_to_drop)

before = len(df)
df = df.dropna().drop_duplicates()
print(f"Rows removed (missing/duplicates): {before - len(df)}")
print("Missing values left:", df.isnull().sum().sum())

In [ ]:
# 2.3 Fix data types
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], utc=True)
df.dtypes

In [ ]:
# 2.4 Remove impossible / invalid records (data-entry errors)
# Valid latitude/longitude range, restricted to the New York City area
lon_min, lon_max = -75.0, -72.0
lat_min, lat_max = 40.0, 42.0

before = len(df)
df = df[
    df["pickup_longitude"].between(lon_min, lon_max) &
    df["dropoff_longitude"].between(lon_min, lon_max) &
    df["pickup_latitude"].between(lat_min, lat_max) &
    df["dropoff_latitude"].between(lat_min, lat_max) &
    df["passenger_count"].between(1, 6) &
    (df["fare_amount"] > 0)
].copy()
print(f"Rows removed (invalid coordinates / passengers / fare): {before - len(df)}")
print("Rows remaining:", len(df))

In [ ]:
# 2.5 Feature engineering: Haversine distance (km)
def haversine(lon_1, lat_1, lon_2, lat_2):
    lon_1, lat_1, lon_2, lat_2 = map(np.radians, [lon_1, lat_1, lon_2, lat_2])
    d_lon = lon_2 - lon_1
    d_lat = lat_2 - lat_1
    a = np.sin(d_lat / 2) ** 2 + np.cos(lat_1) * np.cos(lat_2) * np.sin(d_lon / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df["Distance"] = haversine(df["pickup_longitude"], df["pickup_latitude"],
                           df["dropoff_longitude"], df["dropoff_latitude"]).round(3)

# 2.6 Date / time features
df["Year"]        = df["pickup_datetime"].dt.year
df["Month"]       = df["pickup_datetime"].dt.month
df["Day"]         = df["pickup_datetime"].dt.day
df["DayOfWeek"]   = df["pickup_datetime"].dt.dayofweek      # 0 = Mon ... 6 = Sun
df["Hour"]        = df["pickup_datetime"].dt.hour

# Trips with zero distance are not real rides (pickup == drop-off)
df = df[df["Distance"] > 0].copy()
df[["fare_amount", "Distance", "passenger_count", "Year", "Month", "Day", "DayOfWeek", "Hour"]].head()

## 3. Identify outliers

We first *look* at the outliers (scatter plot, box plots) and quantify them with the **IQR rule** (values outside `Q1 − 1.5·IQR` and `Q3 + 1.5·IQR`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df["Distance"], df["fare_amount"], s=4, alpha=0.4)
axes[0].set(title="Fare vs Distance (before outlier removal)", xlabel="Distance (km)", ylabel="Fare ($)")

sns.boxplot(y=df["fare_amount"], ax=axes[1], color="skyblue")
axes[1].set_title("Box plot - Fare")

sns.boxplot(y=df["Distance"], ax=axes[2], color="salmon")
axes[2].set_title("Box plot - Distance")
plt.tight_layout()
plt.show()

In [ ]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

rows = []
for col in ["fare_amount", "Distance"]:
    lo, hi = iqr_bounds(df[col])
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    rows.append([col, round(lo, 2), round(hi, 2), n_out, round(100 * n_out / len(df), 2)])

pd.DataFrame(rows, columns=["Column", "Lower bound", "Upper bound", "# Outliers", "% of data"])

**Treatment of outliers.** Plenty of long trips (e.g. airport rides) are legitimate, so removing everything beyond 1.5·IQR would throw away real information. We therefore use:
* the more conservative **3·IQR** (“extreme outlier”) limit for fare and distance, and
* a **fare-vs-distance consistency check** to remove suspicious records (very high fare for a tiny distance, or a long distance with a tiny fare).

In [ ]:
before = len(df)

fare_lo, fare_hi = iqr_bounds(df["fare_amount"], k=3)
dist_lo, dist_hi = iqr_bounds(df["Distance"], k=3)
print(f"Extreme limits -> Fare: <= {fare_hi:.2f}, Distance: <= {dist_hi:.2f} km")

df = df[(df["fare_amount"] <= fare_hi) & (df["Distance"] <= dist_hi)]

# Fare / distance inconsistencies
df = df[~((df["fare_amount"] > 50) & (df["Distance"] < 1))]     # expensive but almost no distance
df = df[~((df["fare_amount"] < 5) & (df["Distance"] > 20))]     # long distance but tiny fare

print(f"Rows removed as outliers: {before - len(df)}")
print("Rows remaining:", len(df))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(df["Distance"], df["fare_amount"], s=4, alpha=0.4, color="green")
axes[0].set(title="Fare vs Distance (after outlier removal)", xlabel="Distance (km)", ylabel="Fare ($)")

sns.boxplot(y=df["fare_amount"], ax=axes[1], color="skyblue")
axes[1].set_title("Box plot - Fare (cleaned)")

sns.boxplot(y=df["Distance"], ax=axes[2], color="salmon")
axes[2].set_title("Box plot - Distance (cleaned)")
plt.tight_layout()
plt.show()

## 4. Exploratory analysis (trip patterns)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
df.groupby("Year")["fare_amount"].count().plot(kind="bar", ax=axes[0], color="#1f77b4", title="Trips per year")
df.groupby("Hour")["fare_amount"].count().plot(kind="bar", ax=axes[1], color="#ff7f0e", title="Trips per hour of day")
df.groupby("DayOfWeek")["fare_amount"].count().plot(kind="bar", ax=axes[2], color="#2ca02c", title="Trips per day of week (0 = Mon)")
plt.tight_layout()
plt.show()

In [ ]:
# Average fare by year (fares rose after the 2012 fare increase)
df.groupby("Year")["fare_amount"].mean().round(2).to_frame("Average fare ($)")

## 5. Correlation

Pearson correlation between the numeric variables. We are mainly interested in how each feature relates to `fare_amount`.

In [ ]:
numeric_cols = ["fare_amount", "Distance", "passenger_count", "pickup_longitude", "pickup_latitude",
                "dropoff_longitude", "dropoff_latitude", "Year", "Month", "Day", "DayOfWeek", "Hour"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.4)
plt.title("Correlation matrix")
plt.show()

In [ ]:
corr["fare_amount"].drop("fare_amount").sort_values(ascending=False).to_frame("Correlation with fare_amount")

**Observation:** `Distance` has by far the strongest correlation with `fare_amount`, followed (weakly) by `Year` (fare increases over time). Passenger count, day and month show almost no linear relationship with the fare.

## 6. Train / test split

Features used: distance, passenger count, pickup & drop-off coordinates and date/time features. Target: `fare_amount` (kept in dollars so that RMSE/MAE are directly interpretable).

In [ ]:
features = ["Distance", "passenger_count", "pickup_longitude", "pickup_latitude",
            "dropoff_longitude", "dropoff_latitude", "Year", "Month", "Day", "DayOfWeek", "Hour"]
target = "fare_amount"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 7. Model 1 - Linear Regression

In [ ]:
# Scaling is placed inside a pipeline so it is fitted on the training data only (no data leakage)
lin_reg = make_pipeline(StandardScaler(), LinearRegression())
lin_reg.fit(X_train, y_train)

y_pred_lr_train = lin_reg.predict(X_train)
y_pred_lr = lin_reg.predict(X_test)

coef = pd.Series(lin_reg[-1].coef_, index=features).sort_values(ascending=False)
print("Intercept:", round(lin_reg[-1].intercept_, 3))
coef.to_frame("Standardised coefficient")

## 8. Model 2 - Random Forest Regression

In [ ]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf_reg.fit(X_train, y_train)

y_pred_rf_train = rf_reg.predict(X_train)
y_pred_rf = rf_reg.predict(X_test)

In [ ]:
importances = pd.Series(rf_reg.feature_importances_, index=features).sort_values()
importances.plot(kind="barh", color="teal", title="Random Forest - feature importance")
plt.xlabel("Importance")
plt.show()

## 9. Evaluation and comparison

In [ ]:
def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "Model": name,
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
    }

results = pd.DataFrame([
    evaluate("Linear Regression (train)", y_train, y_pred_lr_train),
    evaluate("Linear Regression (test)",  y_test,  y_pred_lr),
    evaluate("Random Forest (train)",     y_train, y_pred_rf_train),
    evaluate("Random Forest (test)",      y_test,  y_pred_rf),
]).set_index("Model").round(4)
results

In [ ]:
# Test-set comparison chart
test_res = results.loc[["Linear Regression (test)", "Random Forest (test)"]]
test_res.index = ["Linear Regression", "Random Forest"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric, color in zip(axes, ["R2", "RMSE", "MAE"], ["#1f77b4", "#d62728", "#2ca02c"]):
    test_res[metric].plot(kind="bar", ax=ax, color=color, rot=0)
    ax.set_title(f"{metric} (test set)")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom")
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs predicted and residuals
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
lim = [0, max(y_test.max(), y_pred_lr.max(), y_pred_rf.max())]

for col, (name, pred) in enumerate([("Linear Regression", y_pred_lr), ("Random Forest", y_pred_rf)]):
    axes[0, col].scatter(y_test, pred, s=4, alpha=0.4)
    axes[0, col].plot(lim, lim, "r--")
    axes[0, col].set(title=f"{name}: Actual vs Predicted", xlabel="Actual fare ($)", ylabel="Predicted fare ($)")

    sns.histplot(y_test - pred, bins=60, kde=True, ax=axes[1, col])
    axes[1, col].set(title=f"{name}: Residual distribution", xlabel="Actual - Predicted ($)")

plt.tight_layout()
plt.show()

In [ ]:
# Simple fare-vs-distance regression line (linear model fitted on distance only, for visualisation)
lr_simple = LinearRegression().fit(X_train[["Distance"]], y_train)
plt.scatter(X_test["Distance"], y_test, s=4, alpha=0.3, label="Actual")
xs = np.linspace(0, X_test["Distance"].max(), 100).reshape(-1, 1)
plt.plot(xs, lr_simple.predict(pd.DataFrame(xs, columns=["Distance"])), color="red", label="Linear fit")
plt.xlabel("Distance (km)"); plt.ylabel("Fare ($)"); plt.title("Fare vs Distance (test set)"); plt.legend()
plt.show()
print(f"Fare ≈ {lr_simple.intercept_:.2f} + {lr_simple.coef_[0]:.2f} × Distance(km)")

## 10. Predict the fare for a new trip

In [ ]:
def predict_fare(pickup_lat, pickup_lon, drop_lat, drop_lon,
                 passengers=1, when="2015-06-15 18:30", model=rf_reg):
    t = pd.to_datetime(when)
    trip = pd.DataFrame([{
        "Distance": haversine(pickup_lon, pickup_lat, drop_lon, drop_lat),
        "passenger_count": passengers,
        "pickup_longitude": pickup_lon, "pickup_latitude": pickup_lat,
        "dropoff_longitude": drop_lon,  "dropoff_latitude": drop_lat,
        "Year": t.year, "Month": t.month, "Day": t.day, "DayOfWeek": t.dayofweek, "Hour": t.hour,
    }])[features]
    return float(model.predict(trip)[0])

# Example: Times Square -> JFK Airport
args = dict(pickup_lat=40.7580, pickup_lon=-73.9855, drop_lat=40.6413, drop_lon=-73.7781)
print(f"Random Forest predicted fare:     ${predict_fare(**args):.2f}")
print(f"Linear Regression predicted fare: ${predict_fare(**args, model=lin_reg):.2f}")

## 11. Conclusion

* **Pre-processing:** removed index columns, missing values, duplicates and invalid records; built `Distance` (Haversine) and date/time features.
* **Outliers:** identified with scatter/box plots and the IQR rule; removed extreme and inconsistent fare–distance records.
* **Correlation:** `Distance` is by far the most important driver of `fare_amount`; time-based features add a small signal (e.g. `Year`).
* **Models:** compare the table in Section 9. Random Forest normally achieves a higher R² and lower RMSE/MAE than Linear Regression because it captures non-linear effects (pickup/drop-off location, hour of day, fare increases over the years), while Linear Regression is simpler, faster and easier to interpret.
* **Possible improvements:** hyper-parameter tuning (`GridSearchCV`/`RandomizedSearchCV`), gradient-boosting models (XGBoost/LightGBM), distance to airports/landmarks, and a log-transform of the target.